In [23]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from  langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

In [24]:
load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

In [25]:
model = ChatGoogleGenerativeAI(
    model = 'gemini-3.8-flash',
    api_key = api_key
)

In [26]:
class BlogState(TypedDict):

    title: str
    outline: str
    content : str

In [27]:
def create_outline(state: BlogState)-> BlogState:
    title = state['title']
    prompt = f' Generate a detailed outline for a blog on the topic:  {title}'
    outline = model.invoke(prompt).content
    state['outline'] = outline

    return state


In [28]:
def create_blog(state:BlogState)-> BlogState:
    title = state['title']
    outline = state['outline']
    prompt = f'Write a detailed blog on the title: {title} using the following outline\n {outline}'
    content = model.invoke(prompt).content
    state['content'] = content
    return state

In [29]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END) 

workflow = graph.compile()

In [30]:
initial_state = { 'title': 'earth'}
final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'earth', 'outline': [{'type': 'text', 'text': 'Here is a detailed, SEO-friendly blog outline centered on the theme of **Earth**, designed to take the reader on a journey from cosmic wonder to actionable stewardship.\n\n---\n\n# Blog Post Title Ideas:\n* **The Living Marble: Why Earth Is Far More Than Just a Rock in Space**\n* **Planet Earth 101: The Miracle of Our Cosmic Oasis**\n* **The Fragile Giant: Exploring the Beauty, Systems, and Future of Earth**\n\n---\n\n## 1. Introduction\n* **The Hook:** Reference the famous "Pale Blue Dot" photograph or the *Earthrise* image taken by Apollo 8—a tiny, luminous sanctuary in the vast, cold void of space.\n* **The Reality:** We have discovered thousands of exoplanets, yet Earth remains the only known harbor for life.\n* **Thesis Statement:** Earth is not just a platform we stand on; it is a complex, breathing super-organism whose delicate systems make human existence possible.\n* **Overview:** A brief roadmap of the blog post (Origin

In [31]:
print(final_state['outline'])

[{'type': 'text', 'text': 'Here is a detailed, SEO-friendly blog outline centered on the theme of **Earth**, designed to take the reader on a journey from cosmic wonder to actionable stewardship.\n\n---\n\n# Blog Post Title Ideas:\n* **The Living Marble: Why Earth Is Far More Than Just a Rock in Space**\n* **Planet Earth 101: The Miracle of Our Cosmic Oasis**\n* **The Fragile Giant: Exploring the Beauty, Systems, and Future of Earth**\n\n---\n\n## 1. Introduction\n* **The Hook:** Reference the famous "Pale Blue Dot" photograph or the *Earthrise* image taken by Apollo 8—a tiny, luminous sanctuary in the vast, cold void of space.\n* **The Reality:** We have discovered thousands of exoplanets, yet Earth remains the only known harbor for life.\n* **Thesis Statement:** Earth is not just a platform we stand on; it is a complex, breathing super-organism whose delicate systems make human existence possible.\n* **Overview:** A brief roadmap of the blog post (Origin & Location $\\rightarrow$ Dyn

In [33]:
print(final_state['content'])

[{'type': 'text', 'text': '# The Living Marble: Why Earth Is Far More Than Just a Rock in Space\n\nOn Christmas Eve, 1968, astronaut William Anders looked out the window of Apollo 8 as it swung around the desolate, crater-scarred horizon of the Moon. Looming in the distance, contrasting against the pitch-black void of the cosmic abyss, was a swirling orb of brilliant sapphire, blinding white, and subtle terracotta. Anders snapped the photograph known today as *Earthrise*. A little over two decades later, Voyager 1 turned its camera backward from the rim of the solar system to capture Earth as a single, fragile "Pale Blue Dot" suspended in a sunbeam.\n\nThose iconic images permanently altered human consciousness. They revealed our world not as an infinite expanse of resources, but as a solitary, luminous sanctuary adrift in an unforgiving desert of vacuum and radiation.\n\n```\n       "Look again at that dot. That\'s here. That\'s home. That\'s us."\n                              — Carl